Running all these queries in Google Big Query

Company's key risk assumption 2: Older employees may require more medical attention. defining my older to be age 40 and above
- Medical attention could mean one or more of: Higher number of claims, Higher claim costs, More severe claim types, More frequent claims,Higher claim approval rates

In [ ]:
-- 1. Do older employees generate higher claim costs?
SELECT
    CASE
        WHEN ce.Age BETWEEN 18 AND 29 THEN '18-29'
        WHEN ce.Age BETWEEN 30 AND 39 THEN '30-39'
        WHEN ce.Age BETWEEN 40 AND 49 THEN '40-49'
        WHEN ce.Age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '60+'
    END AS Age_Group,
    COUNT(c.Claim_ID) AS Number_of_Claims,
    COUNT(DISTINCT ce.Employee_ID) AS Number_of_Employees,
    ROUND(
        COUNT(c.Claim_ID) /
        COUNT(DISTINCT ce.Employee_ID),
        2
    ) AS Claims_Per_Employee
FROM `my-insurance-project-498821.insurance_dataset.covered_employees` ce
LEFT JOIN `my-insurance-project-498821.insurance_dataset.claims` c
ON ce.Employee_ID = c.Employee_ID
WHERE c.Claim_Type NOT IN (
    'Funeral Benefit',
    'Death Benefit',
    'Maternity'
)
GROUP BY 1
ORDER BY 1;

In [ ]:
-- Code output
Row | Age_Group| Number_of_Claims | Number_of_Employees | Claims_Per_Employee
1      18-29         447                 260                    1.72
2      30-39         694                 398                    1.74
3      40-49         698                 400                    1.75
4      50-59         661                 358                    1.85


Interpretation
The analysis shows a gradual increase in the average number of claims submitted per employee as age increases. 
Employees aged 40–49 and 50–59 recorded the highest claims-per-employee ratio (1.48 claims per employee), 
while employees aged 18–29 recorded the lowest ratio (1.43 claims per employee).
Although the differences between age groups are relatively small, the pattern indicates that older employees tend
to submit slightly more claims than younger employees.

Does this support the company's assumption?
Yes, but only moderately.
The results suggest that older employees make slightly more insurance claims per employee than younger employees, 
providing some support for the assumption that healthcare utilization increases with age. However, the differences
are not substantial enough to conclude that age alone is a strong driver of claim frequency.

Possible Business Explanation:
As employees grow older, they may require more routine medical monitoring and treatment for chronic or age-related 
conditions. These factors can contribute to a gradual increase in healthcare utilization and claim submissions.

Recommendation:
The insurer should continue monitoring claim frequency across age groups and consider preventive health initiatives
for employees aged 40 years and above. Tracking this metric over multiple years would help determine whether the 
observed trend remains consistent.



In [ ]:
-- Analysis 2 — Do older employees generate higher claim costs?
SELECT
  CASE
        WHEN ce.Age BETWEEN 18 AND 29 THEN '18-29'
        WHEN ce.Age BETWEEN 30 AND 39 THEN '30-39'
        WHEN ce.Age BETWEEN 40 AND 49 THEN '40-49'
        WHEN ce.Age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '60+'
  END AS Age_Group,
  SUM(c.Claim_Amount_GHS) AS Claim_Amount,
  AVG(c.Claim_Amount_GHS) AS Avg_Claim_Amount,
FROM `my-insurance-project-498821.insurance_dataset.covered_employees` ce
JOIN `my-insurance-project-498821.insurance_dataset.claims` c ON ce.EMployee_ID = c.Employee_ID
WHERE c.Claim_Type NOT IN (
    'Funeral Benefit',
    'Death Benefit',
    'Maternity'
)
GROUP BY 1
ORDER BY 1;


In [ ]:
Output:
Row | Age_Group | Claim_Amount | Avg_Claim_Amount
1       18-29       7206706       24346.979729729745
2       30-39       10259491      22900.649553571428
3       40-49       11458690      24642.344086021516
4       50-59       10477924      24770.5059101655

Interpretation: 
Employees aged 40–49 generated the highest total claim costs (GHS 11.46 million), while employees aged 50–59 
recorded the highest average claim amount per claim (GHS 24,771). In contrast, employees aged 30–39 had the lowest 
average claim amount despite submitting a relatively high number of claims.
This indicates that while claim frequency increases slightly with age, the cost of individual claims also tends to 
be higher among older employees.

Does this support the company's assumption?
Yes.
The findings indicate that employees aged 40 years and above impose a greater financial burden on the insurer. 
Although the oldest age group did not generate the highest total claim cost, they recorded the highest average 
claim amount, suggesting that medical treatments become more expensive as employees age.

Possible Business Explanation
Older employees are more likely to require specialized treatments, diagnostic procedures, or management of chronic 
medical conditions, all of which typically increase the cost of healthcare services.

Recommendation
The insurer should investigate which medical conditions contribute most to high-value claims among employees aged 
40 years and above. Preventive health programmes and early disease management may help reduce future claim costs.



In [ ]:
-- Analysis 3 — Which claim type generates the highest rates among the age groups
WITH ClaimCounts AS (

  SELECT
    CASE
        WHEN ce.Age BETWEEN 18 AND 29 THEN '18-29'
        WHEN ce.Age BETWEEN 30 AND 39 THEN '30-39'
        WHEN ce.Age BETWEEN 40 AND 49 THEN '40-49'
        WHEN ce.Age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '60+'
    END AS Age_Group,
    c.Claim_Type,
    COUNT(*) AS Number_of_Claims
  FROM `my-insurance-project-498821.insurance_dataset.covered_employees` ce
  JOIN `my-insurance-project-498821.insurance_dataset.claims` c
  ON ce.Employee_ID = c.Employee_ID
  WHERE c.Claim_Type NOT IN (
    'Funeral Benefit',
    'Death Benefit',
    'Maternity'
  )
  GROUP BY Age_Group, Claim_Type
),

RankedClaims AS (

SELECT *,
       ROW_NUMBER() OVER(
           PARTITION BY Age_Group
           ORDER BY Number_of_Claims DESC
       ) AS rn

FROM ClaimCounts
)

SELECT
    Age_Group,
    Claim_Type AS Most_Common_Claim,
    Number_of_Claims

FROM RankedClaims

WHERE rn = 1

ORDER BY Age_Group;

In [ ]:
Output:
Row | Age_Group | Most_Common_Claim | Number_of_Claims
1       18-29          Dental              55
2       30-39      Hospitalization         93
3       40-49      Medical Support         87
4       50-59      Emergency Care          77


Interpretation
The dominant claim type changes noticeably across age groups. Younger employees most frequently submitted Dental 
claims, whereas employees aged 30–39 most commonly submitted Hospitalization claims. Employees aged 40–49 primarily 
claimed Medical Support, while employees aged 50–59 most frequently required Emergency Care.

Does this support the company's assumption?
Yes.
The progression from relatively routine services, such as dental care, among younger employees to hospitalization, 
medical support, and emergency care among older employees suggests that healthcare needs become more intensive 
with age.

Possible Business Explanation
As employees age, the likelihood of acute illnesses and chronic health conditions increases. Consequently, 
older employees may require more specialized and urgent healthcare services than younger employees.

Recommendation
The insurer should consider developing age-specific healthcare programmes. For example:
Preventive dental and wellness initiatives for younger employees.
Chronic disease management programmes for middle-aged employees.
Preventive screenings and emergency risk reduction programmes for older employees.


In [ ]:
-- Analysis 4 — Are claims more likely to be approved for older employees
SELECT
    CASE
        WHEN ce.Age BETWEEN 18 AND 29 THEN '18-29'
        WHEN ce.Age BETWEEN 30 AND 39 THEN '30-39'
        WHEN ce.Age BETWEEN 40 AND 49 THEN '40-49'
        WHEN ce.Age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '60+'
    END AS Age_Group,

    COUNT(c.Claim_ID) AS Total_Claims,

    COUNTIF(c.Claim_Status = 'Approved') AS Approved_Claims,

    ROUND(
        COUNTIF(c.Claim_Status = 'Approved') * 100.0 /
        COUNT(c.Claim_ID),
        2
    ) AS Approval_Rate_Percent

FROM `my-insurance-project-498821.insurance_dataset.covered_employees` ce

JOIN `my-insurance-project-498821.insurance_dataset.claims` c
ON ce.Employee_ID = c.Employee_ID
WHERE c.Claim_Type NOT IN (
    'Funeral Benefit',
    'Death Benefit',
    'Maternity'
  )

GROUP BY Age_Group

ORDER BY Age_Group;

In [ ]:
Output:
| Age Group | Approval Rate |
| --------- | ------------: |
| 18–29     |        66.55% |
| 30–39     |        72.77% |
| 40–49     |        70.11% |
| 50–59     |        65.48% |


Interpretation
Employees aged 30–39 recorded the highest claim approval rate (72.77%), followed by employees aged 40–49 (70.11%). 
Contrary to expectations, employees aged 50–59 had the lowest approval rate (65.48%), slightly below employees 
aged 18–29.

Does this support the company's assumption?
No.
While older employees submitted more claims per employee and incurred higher average claim costs, 
their claims were not more likely to be approved. Therefore, the assumption that older employees receive 
a higher proportion of approved claims is not supported by the available data.

Possible Business Explanation
Several factors could explain this pattern:
Older employees may submit claims that exceed policy limits or are excluded under the policy.
Emergency or complex treatments may require additional documentation, increasing the likelihood of rejection.
Differences in approval rates may reflect policy rules rather than underlying healthcare needs.
** NB:Further investigation into claim rejection reasons would be needed before drawing operational conclusions.

Recommendation:
The insurer should analyze rejected claims for employees aged 50 years and above to identify common rejection 
reasons. If rejections are driven by documentation or policy misunderstandings rather than fraud or ineligibility, 
targeted communication and claims support could improve the customer experience.

In [ ]:
Overall Conclusion for Risk Assumption 2
Based on the analyses, the company's assumption is largely supported, but with important nuances:
1. Employees aged 50–59 submitted the highest number of claims per employee.
2. Employees aged 40–49 generated the highest total and average claim costs.
3. The most common claim types shifted from relatively routine care (Dental) in younger employees to more intensive 
services (Hospitalization, Medical Support, and Emergency Care) in older age groups.
4. However, older employees did not have the highest claim approval rates, indicating that greater healthcare 
utilization does not necessarily translate into a higher proportion of approved claims.

Overall, the evidence suggests that healthcare utilization and financial risk generally increase with age, 
supporting the company's concern that older employees may require more medical attention. Nevertheless, 
the approval-rate analysis highlights the need for further investigation into claim assessment practices and 
reasons for claim denials among older employees.

Company's Risk Assumption 3
Customers with frequent claims may represent higher insurance exposure 
Company would like to find out whether frequent claimants represent higher insurance exposure.
The goal is to determine whether covered employees who submit claims more frequently expose the insurer to greater financial risk.

In [ ]:
To discover and define frequent claimant, 
(i) i will start by finding how many claims each covered employee submitted

SELECT
    Employee_ID,
    COUNT(Claim_ID) AS Number_of_Claims
FROM `my-insurance-project-498821.insurance_dataset.claims`
GROUP BY Employee_ID
ORDER BY Number_of_Claims DESC;


Output:
first 50 rows

Row	Employee_ID	Number_of_Claims
1	EMP01758	6
2	EMP00881	6
3	EMP00042	6
4	EMP01360	5
5	EMP01147	5
11	EMP00850	5
12	EMP00665	5
13	EMP01904	5
14	EMP01739	4
15	EMP01189	4
46	EMP01530	4
47	EMP00419	4
48	EMP00231	4
49	EMP00355	4
50	EMP00354	4


Interpretation:
The maximum number of claims submitted by any covered employee during the two-year period is 6 claims.
This immediately tells us that there are no extreme outliers such as employees making 15 or 20 claims.

In [ ]:
Step 2 to help define frequent claimant
let's see the distribution.
How many employees submitted 1 claim, 2 claims, 3 claims, etc.?

WITH ClaimFrequency AS (

SELECT
    Employee_ID,
    COUNT(Claim_ID) AS Number_of_Claims
FROM `my-insurance-project-498821.insurance_dataset.claims`
GROUP BY Employee_ID

)

SELECT
    Number_of_Claims,

    COUNT(*) AS Number_of_Employees,

    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS Percentage_of_Employees

FROM ClaimFrequency

GROUP BY Number_of_Claims

ORDER BY Number_of_Claims;


Output:
Row	Number_of_Claims	Number_of_Employees	Percentage_of_Employees
1	1	700	49.44
2	2	444	31.36
3	3	192	13.56
4	4	67	4.73
5	5	10	0.71
6	6	3	0.21

Interpretation:
Almost 81% of all covered employees submitted only one or two claims during the two-year period.

Only:
10 employees submitted 5 claims
3 employees submitted 6 claims

This means employees with five or more claims are extremely rare.

In [ ]:
-- Defining types of claimants


WITH ClaimFrequency AS (

SELECT
    Employee_ID,
    COUNT(Claim_ID) AS Number_of_Claims
FROM `my-insurance-project-498821.insurance_dataset.claims`
GROUP BY Employee_ID

)

SELECT

    CASE
        WHEN Number_of_Claims BETWEEN 1 AND 2 THEN 'Low Utilization'
        WHEN Number_of_Claims BETWEEN 3 AND 4 THEN 'Moderate Utilization'
        ELSE 'High Utilization'
    END AS Utilization_Group,

    COUNT(*) AS Number_of_Employees,

    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS Percentage_of_Employees

FROM ClaimFrequency

GROUP BY Utilization_Group

ORDER BY
CASE
    WHEN Utilization_Group = 'Low Utilization' THEN 1
    WHEN Utilization_Group = 'Moderate Utilization' THEN 2
    ELSE 3
END;


Ouptput:
| Utilization Group    | Employees | Percentage |
| -------------------- | --------: | ---------: |
| Low Utilization      |      1144 |     80.79% |
| Moderate Utilization |       259 |     18.29% |
| High Utilization     |        13 |      0.92% |


In [ ]:
calculate some summary statistics to help us choose a threshold for the frequent claimant objectively.

WITH ClaimFrequency AS (

SELECT
    Employee_ID,
    COUNT(Claim_ID) AS Number_of_Claims
FROM `my-insurance-project-498821.insurance_dataset.claims`
GROUP BY Employee_ID

)

SELECT
    COUNT(*) AS Total_Employees,
    MIN(Number_of_Claims) AS Minimum_Claims,
    MAX(Number_of_Claims) AS Maximum_Claims,
    ROUND(AVG(Number_of_Claims),2) AS Average_Claims
FROM ClaimFrequency;


Output:
| Metric          | Value |
| --------------- | ----: |
| Total Employees |  1416 |
| Minimum Claims  |     1 |
| Maximum Claims  |     6 |
| Average Claims  |  1.77 |


Interpretation:
The average employee submitted:

1.77 claims over two years.

That means someone submitting 5 or 6 claims is claiming nearly three times the average.

From a risk management perspective, that's a meaningful difference.

In [ ]:
Now overall interpretaion for defining the frequent claimant

Analysis of claim frequency over the two-year period shows that the average covered employee submitted 1.77 claims,
with the maximum observed frequency being six claims.

Approximately 81% of covered employees submitted only one or two claims, indicating that low claim frequency is
the norm within the insured population. In contrast, only 13 employees (0.9%) submitted five or more claims, 
making them a distinct minority.

Based on this distribution, covered employees submitting five or more claims are classified as frequent claimants 
for the purposes of this analysis.

In [ ]:
Does this answer the business assumption?

Not yet.
This analysis only defines who the frequent claimants are.
We still need to determine whether they actually represent higher insurance exposure.

In [ ]:
--Analysis 2
Do frequent claimants generate higher claim costs than low- and moderate-utilization claimants?

Code:
WITH ClaimFrequency AS (

SELECT

    Employee_ID,

    COUNT(Claim_ID) AS Number_of_Claims

FROM `my-insurance-project-498821.insurance_dataset.claims`

GROUP BY Employee_ID

),

UtilizationGroups AS (

SELECT

    Employee_ID,

    Number_of_Claims,

    CASE

        WHEN Number_of_Claims BETWEEN 1 AND 2 THEN 'Low Utilization'

        WHEN Number_of_Claims BETWEEN 3 AND 4 THEN 'Moderate Utilization'

        ELSE 'High Utilization'

    END AS Utilization_Group

FROM ClaimFrequency

)

SELECT

    ug.Utilization_Group,

    COUNT(DISTINCT ug.Employee_ID) AS Number_of_Employees,

    COUNT(c.Claim_ID) AS Total_Claims,

    SUM(c.Claim_Amount_GHS) AS Total_Claim_Amount,

    ROUND(AVG(c.Claim_Amount_GHS),2) AS Average_Claim_Amount,
    ROUND(
        SUM(c.Claim_Amount_GHS) /
        COUNT(DISTINCT ug.Employee_ID),
        2
    ) AS Claim_Cost_Per_Employee

FROM UtilizationGroups ug

JOIN `my-insurance-project-498821.insurance_dataset.claims` c

ON ug.Employee_ID = c.Employee_ID

GROUP BY ug.Utilization_Group

ORDER BY

CASE

WHEN Utilization_Group='Low Utilization' THEN 1

WHEN Utilization_Group='Moderate Utilization' THEN 2

ELSE 3

END;



Output:
| Utilization Group    | Employees | Total Claims | Total Claim Amount (₵) | Average Claim Amount (₵) | Claim Cost per Employee (₵) |
| -------------------- | --------: | -----------: | ---------------------: | -----------------------: | --------------------------: |
| Low Utilization      |     1,144 |        1,588 |             44,054,247 |                27,741.97 |                   38,508.96 |
| Moderate Utilization |       259 |          844 |             23,569,077 |                27,925.45 |                   91,000.30 |
| High Utilization     |        13 |           68 |              1,598,520 |                23,507.65 |                  122,963.08 |




Interpretation:
The results show that Low Utilization claimants generated the highest total claim amount (₵44.05 million). 
However, this is expected because they represent over 80% of all covered employees.

When the results are adjusted for the number of employees in each group, a different pattern emerges:
High Utilization claimants had the highest claim cost per employee (₵122,963.08).
Moderate Utilization claimants followed with ₵91,000.30 per employee.
Low Utilization claimants had the lowest cost per employee at ₵38,508.96.

Interestingly, the average amount per individual claim was not highest among High Utilization claimants. Their average 
claim amount (₵23,507.65) was lower than that of the Low and Moderate Utilization groups.

This suggests that the higher financial exposure associated with High Utilization claimants is driven primarily by the 
frequency of claims, rather than the cost of each individual claim.


What it means
The findings suggest that repeated healthcare utilization, rather than unusually expensive individual treatments, 
is the primary driver of increased insurance exposure. Frequent claimants may require ongoing medical care, follow-up 
treatments, or management of chronic conditions, resulting in multiple claims over the policy period.


Does this support the assumption?

Yes, strongly.
Although High Utilization claimants represent less than 1% of covered employees, each employee in this group costs 
the insurer substantially more than employees in the other utilization groups. This indicates that frequent claimants 
represent higher insurance exposure because they submit claims more often, even if each claim is not individually more 
expensive.


Recommendations
Monitor High Utilization claimants as a distinct risk segment.
Investigate the medical conditions or claim types driving repeated claims.
Develop targeted wellness or disease management programmes to reduce repeated utilization.
Consider incorporating claim frequency into future risk assessment and pricing models.

Limitation

This analysis measures claim frequency and claim costs but does not account for:
Premiums paid by each claimant.
Medical diagnoses or reasons for repeated claims.
Whether claims were approved or rejected.

These factors should be examined before making underwriting or pricing decisions.

Premiums paid by each claimant is very important because it will directly tests whether frequent claimants
 are financially profitable or costly to the insurer.
Unfortunately, there is not a direct link between premium payments table and corvered_employees.. 




In [ ]:
Continuing to the next risk assumption
-- Organizations with high claim-to-premium ratios may pose greater financial risk 
to the company 

